In [0]:
CATALOG = "mvp_mmgd"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

df_empreendimento = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.empreendimento_gd")
df_tecnica = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.tecnica_fotovoltaica")

print(f"empreendimento_gd: {df_empreendimento.count():,} linhas")
print(f"tecnica_fotovoltaica: {df_tecnica.count():,} linhas")

In [0]:
from pyspark.sql.functions import (
    sequence, explode, date_format, year, month, quarter, dayofmonth,
    lit, min as spark_min, max as spark_max, col, expr
)
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, StringType, BooleanType

# Intervalo real, excluindo a data sentinela 1900-01-01
data_min = (
    df_empreendimento
    .filter(col("DthAtualizaCadastralEmpreend") != "1900-01-01")
    .select(spark_min("DthAtualizaCadastralEmpreend"))
    .first()[0]
)
data_max = df_empreendimento.select(spark_max("DthAtualizaCadastralEmpreend")).first()[0]

print(f"Intervalo real: {data_min} até {data_max}")

# Gera uma linha para cada dia do intervalo
df_calendario = (
    spark.createDataFrame([(data_min, data_max)], ["inicio", "fim"])
    .select(explode(sequence(col("inicio"), col("fim"), expr("interval 1 day"))).alias("data"))
)

d_tempo = (
    df_calendario
    .withColumn("sk_tempo", date_format(col("data"), "yyyyMMdd").cast("int"))
    .withColumn("ano", year(col("data")))
    .withColumn("mes", month(col("data")))
    .withColumn("nome_mes", date_format(col("data"), "MMMM"))
    .withColumn("trimestre", quarter(col("data")))
    .withColumn("dia", dayofmonth(col("data")))
    .withColumn("flag_pos_lei_14300", (col("data") >= lit("2022-01-06")).cast("boolean"))
)

# Membro desconhecido — para os 23 registros com data sentinela.
# Schema declarado explicitamente, porque uma linha quase toda com None
# não dá pista suficiente para o Spark inferir o tipo sozinho.
schema_membro_desconhecido = StructType([
    StructField("sk_tempo", IntegerType(), True),
    StructField("data", DateType(), True),
    StructField("ano", IntegerType(), True),
    StructField("mes", IntegerType(), True),
    StructField("nome_mes", StringType(), True),
    StructField("trimestre", IntegerType(), True),
    StructField("dia", IntegerType(), True),
    StructField("flag_pos_lei_14300", BooleanType(), True),
])

membro_desconhecido = spark.createDataFrame(
    [(-1, None, None, None, "Desconhecida", None, None, None)],
    schema=schema_membro_desconhecido
)

d_tempo_final = d_tempo.unionByName(membro_desconhecido)

print(f"Total de dias gerados (+ membro desconhecido): {d_tempo_final.count():,}")
display(d_tempo_final.limit(5))
display(d_tempo_final.filter(col("sk_tempo") == -1))

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

#Agora vamos para a segunda dimensão: distribuidoras
d_distribuidora = (
    df_empreendimento
    .select("SigAgente", "NomAgente", "NumCNPJDistribuidora")
    .distinct()
    .withColumn("sk_distribuidora", monotonically_increasing_id())
)

print(f"Total de distribuidoras únicas: {d_distribuidora.count():,}")
display(d_distribuidora.limit(10))

In [0]:
#Identificado distribuidos que não são do Sul: Equatorial e Enel, fazendo novo filtro para descobrir o motivo

qtd_por_distribuidora_suspeita = (
    df_empreendimento
    .filter(col("SigAgente").isin(["EQUATORIAL GO", "ENEL CE"]))
    .groupBy("SigAgente", "SigUF", "NomMunicipio", "CodMunicipioIbge")
    .count()
    .orderBy("count", ascending=False)
)

display(qtd_por_distribuidora_suspeita.limit(20))

In [0]:
qtd_total_emp = df_empreendimento.count()
#Ententendo qual o percentual de distribuidoras que estão incompatíveis com a cidade UF
qtd_suspeita = df_empreendimento.filter(col("SigAgente").isin(["EQUATORIAL GO", "ENEL CE"])).count()
print(f"Registros com distribuidora possivelmente incompatível com a UF: {qtd_suspeita:,} de {qtd_total_emp:,} ({qtd_suspeita/qtd_total_emp:.3%})")

In [0]:
#Agora vamos para a dimensão d_localidade
d_localidade = (
    df_empreendimento
    .select("CodMunicipioIbge", "NomMunicipio", "SigUF", "NomRegiao", "CodRegiao")
    .distinct()
    .withColumn("sk_localidade", monotonically_increasing_id())
)

print(f"Total de localidades únicas: {d_localidade.count():,}")
display(d_localidade.limit(10))

In [0]:
#Dimensão d_classificacao_consumo

d_classificacao_consumo = (
    df_empreendimento
    .select("CodClasseConsumo", "DscClasseConsumo", "DscSubGrupoTarifario")
    .distinct()
    .withColumn("sk_classificacao_consumo", monotonically_increasing_id())
)

print(f"Total de combinações únicas: {d_classificacao_consumo.count():,}")
display(d_classificacao_consumo.limit(15))

In [0]:
#Dimensão d_modalidade
d_modalidade = (
    df_empreendimento
    .select("DscModalidadeHabilitado")
    .distinct()
    .withColumn("sk_modalidade", monotonically_increasing_id())
)

print(f"Total de modalidades únicas: {d_modalidade.count():,}")
display(d_modalidade)

In [0]:
#Dimensão d_fonte
d_fonte = (
    df_empreendimento
    .select("SigTipoGeracao", "DscFonteGeracao", "DscPorte")
    .distinct()
    .withColumn("sk_fonte", monotonically_increasing_id())
)

print(f"Total de combinações fonte/porte únicas: {d_fonte.count():,}")
display(d_fonte)

In [0]:
#Entendo e quantificando os nulos

total = df_empreendimento.count()

qtd_null_subgrupo = df_empreendimento.filter(col("CodSubGrupoTarifario").isNull()).count()
qtd_null_modalidade = df_empreendimento.filter(col("SigModalidadeEmpreendimento").isNull()).count()

print(f"CodSubGrupoTarifario nulo: {qtd_null_subgrupo:,} de {total:,} ({qtd_null_subgrupo/total:.2%})")
print(f"SigModalidadeEmpreendimento nulo: {qtd_null_modalidade:,} de {total:,} ({qtd_null_modalidade/total:.2%})")

#Como o valor ficou próximo de 100% vou alterar as células para excluir as colunas.

In [0]:
d_fabricante = (
    df_tecnica
    .select(col("NomFabricanteModulo").alias("nome_fabricante"))
    .union(df_tecnica.select(col("NomFabricanteInversor").alias("nome_fabricante")))
    .filter(col("nome_fabricante").isNotNull())
    .distinct()
    .withColumn("sk_fabricante", monotonically_increasing_id())
)

print(f"Total de fabricantes únicos: {d_fabricante.count():,}")
display(d_fabricante.limit(15))

In [0]:
#Verificando a concentração de fabricantes, 25736 é um numero irreal...

from pyspark.sql.functions import sum as spark_sum, desc

freq_fabricante = (
    df_tecnica.select(col("NomFabricanteModulo").alias("nome_fabricante"))
    .union(df_tecnica.select(col("NomFabricanteInversor").alias("nome_fabricante")))
    .filter(col("nome_fabricante").isNotNull())
    .groupBy("nome_fabricante")
    .count()
)

total_ocorrencias = freq_fabricante.agg(spark_sum("count")).first()[0]
qtd_raros = freq_fabricante.filter(col("count") <= 5).count()
cobertura_top50 = (
    freq_fabricante.orderBy(desc("count")).limit(50)
    .agg(spark_sum("count")).first()[0]
)

print(f"Total de ocorrências (módulo + inversor): {total_ocorrencias:,}")
print(f"Fabricantes com até 5 ocorrências: {qtd_raros:,} de {freq_fabricante.count():,}")
print(f"Cobertura dos 50 maiores: {cobertura_top50/total_ocorrencias:.2%} das ocorrências")

display(freq_fabricante.orderBy(desc("count")).limit(30))
display(freq_fabricante.filter(col("nome_fabricante").isin(["RISEN", "RISAN"])))

In [0]:
from pyspark.sql.functions import when, desc, lit, monotonically_increasing_id

# Frequência de cada nome, somando uso como módulo e como inversor
freq_fabricante = (
    df_tecnica.select(col("NomFabricanteModulo").alias("nome_fabricante"))
    .union(df_tecnica.select(col("NomFabricanteInversor").alias("nome_fabricante")))
    .filter(col("nome_fabricante").isNotNull())
    .groupBy("nome_fabricante")
    .count()
)

# Os 50 nomes mais frequentes (cobrem 81,48% das ocorrências)
top50 = (
    freq_fabricante
    .orderBy(desc("count"))
    .limit(50)
    .select("nome_fabricante")
    .withColumn("is_top50", lit(True))
)

d_fabricante = (
    freq_fabricante
    .join(top50, on="nome_fabricante", how="left")
    .withColumn(
        "grupo_fabricante",
        when(col("is_top50"), col("nome_fabricante")).otherwise(lit("OUTROS"))
    )
    .select("nome_fabricante", "grupo_fabricante")
    .withColumn("sk_fabricante", monotonically_increasing_id())
)

print(f"Total de fabricantes na dimensão: {d_fabricante.count():,}")
print(f"Grupos distintos (top 50 + OUTROS): {d_fabricante.select('grupo_fabricante').distinct().count()}")
display(d_fabricante.filter(col("grupo_fabricante") != "OUTROS").limit(10))
display(d_fabricante.filter(col("grupo_fabricante") == "OUTROS").limit(10))

In [0]:
from pyspark.sql.functions import when, desc, lit, monotonically_increasing_id

# Frequência de cada nome, somando uso como módulo e como inversor
freq_fabricante = (
    df_tecnica.select(col("NomFabricanteModulo").alias("nome_fabricante"))
    .union(df_tecnica.select(col("NomFabricanteInversor").alias("nome_fabricante")))
    .filter(col("nome_fabricante").isNotNull())
    .groupBy("nome_fabricante")
    .count()
)

# Os 50 nomes mais frequentes (cobrem 81,48% das ocorrências)
top50 = (
    freq_fabricante
    .orderBy(desc("count"))
    .limit(50)
    .select("nome_fabricante")
    .withColumn("is_top50", lit(True))
)

d_fabricante = (
    freq_fabricante
    .join(top50, on="nome_fabricante", how="left")
    .withColumn(
        "grupo_fabricante",
        when(col("is_top50"), col("nome_fabricante")).otherwise(lit("OUTROS"))
    )
    .select("nome_fabricante", "grupo_fabricante")
    .withColumn("sk_fabricante", monotonically_increasing_id())
)

print(f"Total de fabricantes na dimensão: {d_fabricante.count():,}")
print(f"Grupos distintos (top 50 + OUTROS): {d_fabricante.select('grupo_fabricante').distinct().count()}")
display(d_fabricante.filter(col("grupo_fabricante") != "OUTROS").limit(10))
display(d_fabricante.filter(col("grupo_fabricante") == "OUTROS").limit(10))

In [0]:
#Gravando as dimensões como tabelas Delta
from pyspark.sql.functions import current_timestamp, lit

dimensoes = {
    "d_tempo": d_tempo_final,
    "d_distribuidora": d_distribuidora,
    "d_localidade": d_localidade,
    "d_classificacao_consumo": d_classificacao_consumo,
    "d_modalidade": d_modalidade,
    "d_fonte": d_fonte,
    "d_fabricante": d_fabricante,
}

for nome_tabela, df_dim in dimensoes.items():
    (
        df_dim
        .withColumn("_camada", lit("gold"))
        .withColumn("_dth_carga", current_timestamp())
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.{nome_tabela}")
    )
    qtd = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.{nome_tabela}").count()
    print(f"{CATALOG}.{SCHEMA_GOLD}.{nome_tabela} gravada — {qtd:,} linhas")

In [0]:
from pyspark.sql.functions import col, when, date_format, expr

# join com a dimensão gravada para trazer a sk (eqNullSafe para casar os nulos)
def join_dim(df, tabela, chaves, sk):
    dim = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.{tabela}") \
        .select(sk, *[col(c).alias("d_" + c) for c in chaves])
    cond = [col(c).eqNullSafe(col("d_" + c)) for c in chaves]
    return df.join(dim, cond, "left").drop(*["d_" + c for c in chaves])

f = df_empreendimento
f = join_dim(f, "d_distribuidora", ["SigAgente", "NomAgente", "NumCNPJDistribuidora"], "sk_distribuidora")
f = join_dim(f, "d_localidade", ["CodMunicipioIbge", "NomMunicipio", "SigUF", "NomRegiao", "CodRegiao"], "sk_localidade")
f = join_dim(f, "d_classificacao_consumo", ["CodClasseConsumo", "DscClasseConsumo", "DscSubGrupoTarifario"], "sk_classificacao_consumo")
f = join_dim(f, "d_modalidade", ["DscModalidadeHabilitado"], "sk_modalidade")
f = join_dim(f, "d_fonte", ["SigTipoGeracao", "DscFonteGeracao", "DscPorte"], "sk_fonte")

# data 1900-01-01 (sentinela) ou nula -> membro desconhecido
dt = col("DthAtualizaCadastralEmpreend")
f = f.withColumn("sk_tempo",
    when(dt.isNull() | (dt == "1900-01-01"), -1)
    .otherwise(date_format(dt, "yyyyMMdd").cast("int")))

f_conexao_mmgd = f.select(
    col("CodEmpreendimento").alias("cod_empreendimento"),
    "sk_tempo", "sk_distribuidora", "sk_localidade",
    "sk_classificacao_consumo", "sk_modalidade", "sk_fonte",
    col("MdaPotenciaInstaladaKW").alias("potencia_instalada_kw"),
    expr("try_cast(QtdUCRecebeCredito AS INT)").alias("qtd_uc_recebe_credito"))

# checagens
print("linhas:", f_conexao_mmgd.count())
for c in ["sk_tempo", "sk_distribuidora", "sk_localidade", "sk_classificacao_consumo", "sk_modalidade", "sk_fonte"]:
    print(c, "nulos:", f_conexao_mmgd.filter(col(c).isNull()).count())
print("sk_tempo = -1:", f_conexao_mmgd.filter(col("sk_tempo") == -1).count())

display(f_conexao_mmgd.limit(5))

In [0]:
from pyspark.sql.functions import lit, current_timestamp, split

f_conexao_mmgd.withColumn("_camada", lit("gold")) \
    .withColumn("_dth_carga", current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.f_conexao_mmgd")

print("f_conexao_mmgd gravada:", spark.table(f"{CATALOG}.{SCHEMA_GOLD}.f_conexao_mmgd").count(), "linhas")

# UF que aparece no código do empreendimento x UF cadastrada
uf_codigo = split(col("CodEmpreendimento"), "\\.").getItem(1)
display(
    df_empreendimento.groupBy(uf_codigo.alias("uf_no_codigo"), "SigUF")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
#Montando f_tecnica_fv, gravando e montando

from pyspark.sql.functions import col, expr, lit, current_timestamp

d_fab = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.d_fabricante")
d_mod = d_fab.select(col("nome_fabricante").alias("fab_mod"), col("sk_fabricante").alias("sk_fabricante_modulo"))
d_inv = d_fab.select(col("nome_fabricante").alias("fab_inv"), col("sk_fabricante").alias("sk_fabricante_inversor"))

f_tecnica_fv = df_tecnica \
    .join(d_mod, df_tecnica.NomFabricanteModulo == d_mod.fab_mod, "left") \
    .join(d_inv, df_tecnica.NomFabricanteInversor == d_inv.fab_inv, "left") \
    .select(
        col("CodGeracaoDistribuida").alias("cod_empreendimento"),
        "sk_fabricante_modulo", "sk_fabricante_inversor",
        col("QtdModulos").alias("qtd_modulos"),
        col("MdaAreaArranjo").alias("area_arranjo"),
        col("MdaPotenciaModulos").alias("potencia_modulos_kw"),
        col("MdaPotenciaInversores").alias("potencia_inversores_kw"),
        expr("try_divide(MdaPotenciaModulos, MdaPotenciaInversores)").alias("fator_dimensionamento"))

f_tecnica_fv.withColumn("_camada", lit("gold")).withColumn("_dth_carga", current_timestamp()) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.f_tecnica_fv")

print("f_tecnica_fv gravada:", spark.table(f"{CATALOG}.{SCHEMA_GOLD}.f_tecnica_fv").count(), "linhas")
display(spark.table(f"{CATALOG}.{SCHEMA_GOLD}.f_tecnica_fv").select("fator_dimensionamento").summary())

In [0]:
%sql
--Gravando a descrição de cada tabela Gold
COMMENT ON TABLE mvp_mmgd.gold.f_conexao_mmgd IS 'Fato de conexões MMGD na Região Sul. Grão: um empreendimento. Fonte: ANEEL.';
COMMENT ON TABLE mvp_mmgd.gold.f_tecnica_fv IS 'Fato técnica dos sistemas fotovoltaicos. Grão: um sistema. Inclui fator de dimensionamento (CC/CA).';
COMMENT ON TABLE mvp_mmgd.gold.d_tempo IS 'Calendário diário de 2009 a 2026. sk_tempo = -1 representa data desconhecida.';
COMMENT ON TABLE mvp_mmgd.gold.d_distribuidora IS 'Distribuidoras de energia (sigla, nome e CNPJ).';
COMMENT ON TABLE mvp_mmgd.gold.d_localidade IS 'Municípios da Região Sul com UF e região.';
COMMENT ON TABLE mvp_mmgd.gold.d_classificacao_consumo IS 'Classe de consumo e subgrupo tarifário.';
COMMENT ON TABLE mvp_mmgd.gold.d_modalidade IS 'Modalidade do empreendimento (autoconsumo local, remoto, compartilhada etc.).';
COMMENT ON TABLE mvp_mmgd.gold.d_fonte IS 'Fonte de geração e porte (micro ou minigeração).';
COMMENT ON TABLE mvp_mmgd.gold.d_fabricante IS 'Fabricantes de módulos e inversores. grupo_fabricante agrupa fora do top 50 como OUTROS.';

In [0]:
#Determinando os valores máximo e mínimos

g = f"{CATALOG}.{SCHEMA_GOLD}"
display(spark.table(f"{g}.f_conexao_mmgd").select("potencia_instalada_kw", "qtd_uc_recebe_credito").summary("min", "max"))
display(spark.table(f"{g}.f_tecnica_fv").select("qtd_modulos", "area_arranjo", "potencia_modulos_kw", "potencia_inversores_kw").summary("min", "max"))
display(spark.table(f"{g}.d_classificacao_consumo").select("DscClasseConsumo").distinct())
display(spark.table(f"{g}.d_modalidade"))
display(spark.table(f"{g}.d_fonte"))